# Opening the published ECMWF store, and realizing a day from it

Three things, in order:

1. **Open** the virtual Icechunk store on source.coop — anonymously, no credentials at all
2. **Realize a slice by hand** — what "realization" actually means, at a scale that runs in this notebook
3. **Realize one day with Frisky** — the production path, and how to choose the variable set

## What the store is

| | |
|---|---|
| endpoint | `https://data.source.coop` |
| bucket / prefix | `e4drr-project` / `forecasts/ecmwf_ifs_ens_aws_s3_icechunk_vd` |
| groups | `{0p4,49r1,50r1}` × `{00z,06z,12z,18z}` |
| contents | **virtual chunk references**, not data |

Every chunk is a `(url, offset, length)` triplet pointing into a public GRIB file on
`s3://ecmwf-forecasts`. The whole store is metadata standing in for ~620 TB of GRIB, so
**opening it is free** and reading a field fetches bytes from AWS on demand.

That is also the cost model to keep in mind: a GRIB message is atomic, so reading a
small box still pulls the global 721×1440 message behind it. You pay for messages, not
for the area you keep.

## Install

```bash
pip install icechunk==2.1.1 zarr==3.2.1 xarray==2026.7.0 gribberish==1.6.0 \
            numpy==2.5.1 matplotlib
```

`gribberish` is not optional — it registers the Zarr v3 codec that decodes a GRIB message
out of the bytes the virtual reference points at. Without importing it the reads fail.

## 0. The `AWS_*` trap — run this before anything touches the store

The store is read **anonymously**, but its virtual chunks live on AWS. If the icechunk
virtual-chunk client sees stray `AWS_ENDPOINT_URL` / `AWS_*` variables — which any shell
configured for Ceph or source.coop *writing* will have — it tries to resolve a host that
does not exist for `s3://ecmwf-forecasts`.

The object-store config is cached **process-wide on first chunk fetch**, so unsetting it
later does not help. Pop it first, in this kernel, before any import touches the store.

In [1]:
import os

_stripped = {k: os.environ.pop(k) for k in list(os.environ) if k.startswith("AWS_")}
print("popped AWS_* :", sorted(_stripped) or "(none were set)")

popped AWS_* : (none were set)


## 1. Open the store

Two pieces of configuration matter:

- **`anonymous=True, from_env=False`** on the storage — reading a public source.coop
  repository needs no credentials. Credentials are only ever needed to *write*. Reaching
  for a token here is a habit worth breaking: it was done once, and the read then failed
  when the token expired, for something that never needed it.
- **`authorize_virtual_chunk_access`** — icechunk refuses to follow a virtual reference
  into a bucket you have not named. Without it you get
  `InvalidInputError: a virtual chunk ... resolves to the url prefix s3://ecmwf-forecasts/`.
  The store opens fine; the *read* is what fails, which is why it surprises people.

`force_path_style=True` is required by source.coop's S3 gateway.

In [2]:
import icechunk
import xarray as xr
import numpy as np
import gribberish.zarr  # noqa: F401 -- registers the Zarr v3 GRIB codec

SRC_ENDPOINT = "https://data.source.coop"
SRC_BUCKET   = "e4drr-project"
SRC_PREFIX   = "forecasts/ecmwf_ifs_ens_aws_s3_icechunk_vd"
CONTAINER    = "s3://ecmwf-forecasts/"

storage = icechunk.s3_storage(
    bucket=SRC_BUCKET, prefix=SRC_PREFIX, endpoint_url=SRC_ENDPOINT,
    region="us-east-1", anonymous=True, from_env=False, force_path_style=True,
)
auth = icechunk.containers_credentials({CONTAINER: icechunk.s3_anonymous_credentials()})

# Manifest preload off: the manifests are large and we only want a handful of
# chunks. Preloading them costs seconds per open for no benefit here.
cfg = icechunk.RepositoryConfig.default()
cfg.manifest = icechunk.ManifestConfig(
    preload=icechunk.ManifestPreloadConfig(max_total_refs=0, max_arrays_to_scan=0))

repo = icechunk.Repository.open(storage, config=cfg, authorize_virtual_chunk_access=auth)
session = repo.readonly_session("main")
print("snapshot:", session.snapshot_id)

snapshot: CSG8BX0ZMZMF8Q09KJG0


### What is in it

Twelve groups, one per `(era, cycle)`. Each has its **own time and step axis** — `06z`
and `18z` are short forecasts that stop at +144 h (49 steps), while `00z` and `12z` run
to +360 h (85 steps). That difference matters later.

In [3]:
import zarr

root = zarr.open_group(session.store, mode="r")
print(f"{'group':<12} {'dates':>6} {'steps':>6}  {'first':>10}  {'last':>10}")
total = 0
for era in sorted(root.group_keys()):
    for run in sorted(root[era].group_keys()):
        g = root[era][run]
        t = g["time"][:]
        d = np.datetime64("1970-01-01") + t.astype("timedelta64[h]")
        total += len(t)
        print(f"{era}/{run:<7} {len(t):>6} {g['step'].shape[0]:>6}  "
              f"{str(d.min())[:10]}  {str(d.max())[:10]}")
print(f"\ntotal date-cycles: {total}")

group         dates  steps       first        last
0p4/00z        401     85  2023-01-18  2024-02-28
0p4/06z        400     49  2023-01-18  2024-02-27
0p4/12z        399     85  2023-01-18  2024-02-27
0p4/18z        399     49  2023-01-18  2024-02-27
49r1/00z        804     85  2024-02-29  2026-05-12
49r1/06z        804     49  2024-02-28  2026-05-11
49r1/12z        804     85  2024-02-28  2026-05-11
49r1/18z        804     49  2024-02-28  2026-05-11
50r1/00z        103     85  2026-05-13  2026-08-23
50r1/06z        104     49  2026-05-12  2026-08-23
50r1/12z        104     85  2026-05-12  2026-08-23
50r1/18z        104     49  2026-05-12  2026-08-23

total date-cycles: 5230


### How big is the thing you just opened?

That open took a second or two and downloaded almost nothing. Here is what it addresses.

Three different numbers get called "the size" of this store, and they differ by five
orders of magnitude — so it is worth being explicit about which one you mean:

| | | how it is obtained |
|---|---|---|
| **the store** | ~76 GB | the manifests: the `(url, offset, length)` triplets themselves |
| **the GRIB it virtualizes** | **~1.9 PB** | measured from the published file sizes — see below |
| **the array space** | **9.42 PB** | decoded `float32` volume the arrays declare — computed below |

**The GRIB figure, measured rather than quoted.** Summing what ECMWF actually publishes
under `ifs/{0p25,0p4-beta}/{enfo,oper}` for all four cycles:

| era | per day, all 4 cycles | days | total |
|---|---|---|---|
| `0p4` | 0.725 TB (`20230601`) | 407 | 295 TB |
| `0p25` | 1.73 TB (`20250115`), 1.81 TB (`20260823`) | 907 | 1,605 TB |
| | | **1,314** | **≈ 1.9 PB** |

A single 00z cycle is ~0.57 TB/day at 0.25°, which is why an 00z-only figure comes out
near 620 TB for the archive. This store carries **all four cycles**, so the number that
applies to it is ~1.9 PB, not that one.

The array space is larger still (9.42 PB) because GRIB is compressed and packed at reduced
precision, while the arrays declare `float32`. Neither number is "wrong" — they measure
different things.

The cell below computes the array space from shapes alone. It reads no data: shape × dtype
is metadata, so this is arithmetic on what is already in memory.

One caveat, stated because it inflates that number: arrays are declared at the **full era
shape**, and some of that space is legitimately empty. `49r1` carries a 13-level superset
in which 9-level dates leave four levels unwritten, and unwritten chunks read back as NaN.
So it is the size of the addressable array space, not a claim that every cell holds a
measurement.

In [4]:
STORE_MANIFEST_BYTES = 76.40e9   # measured 2026-09-01 by listing the published prefix
GRIB_BYTES           = 1.90e15   # measured from ECMWF file sizes -- re-derive with the cell below

total_bytes = total_chunks = 0
print(f"{'group':<12} {'arrays':>7} {'TB':>9} {'virtual chunks':>16}")
for era in sorted(root.group_keys()):
    for run in sorted(root[era].group_keys()):
        g = root[era][run]
        gb = nch = nvar = 0
        for name in g.array_keys():
            a = g[name]
            if a.ndim < 4:            # coordinate axes, not fields
                continue
            nvar += 1
            gb  += int(np.prod(a.shape)) * a.dtype.itemsize
            nch += int(np.prod([-(-s // c) for s, c in zip(a.shape, a.chunks)]))
        total_bytes += gb
        total_chunks += nch
        print(f"{era}/{run:<7} {nvar:>7} {gb/1e12:>9.1f} {nch:>16,}")

print(f"\naddressable array space  : {total_bytes/1e15:>6.2f} PB   ({total_bytes/1e12:,.0f} TB)")
print(f"GRIB virtualized         : {GRIB_BYTES/1e15:>6.2f} PB   (measured, see above)")
print(f"the store on disk        : {STORE_MANIFEST_BYTES/1e9:>6.0f} GB   of manifests")
print(f"virtual chunk references : {total_chunks:,}")
print(f"\nleverage: {GRIB_BYTES/STORE_MANIFEST_BYTES:,.0f}x GRIB bytes addressed per byte stored")

group         arrays        TB   virtual chunks
0p4/00z          19     234.3      144,281,805
0p4/06z          19     134.7       82,966,800
0p4/12z          19     233.1      143,562,195
0p4/18z          19     134.4       82,759,383
49r1/00z          59    2417.2      582,051,780
49r1/06z          59    1393.5      335,535,732
49r1/12z          59    2417.2      582,051,780
49r1/18z          57    1376.8      331,517,340
50r1/00z          54     341.2       82,156,920
50r1/06z          54     198.6       47,820,864
50r1/12z          54     344.5       82,954,560
50r1/18z          54     198.6       47,820,864

addressable array space  :   9.42 PB   (9,424 TB)
GRIB virtualized         :   1.90 PB   (measured, see above)
the store on disk        :     76 GB   of manifests
virtual chunk references : 2,545,480,023

leverage: 24,869x GRIB bytes addressed per byte stored


In [5]:
# Re-derive the GRIB figure from what ECMWF actually publishes. Anonymous S3
# LIST, ~8 requests, no data. The bucket throttles with 503 at surprisingly low
# rates, hence the retries -- see SCALING.md.
import urllib.request, re

def grib_bytes_for_day(date, path="ifs/0p25"):
    """Sum the published bytes for one date across all four cycles."""
    total = 0
    for run in ("00", "06", "12", "18"):
        for stream in ("enfo", "oper"):
            url = ("https://ecmwf-forecasts.s3.amazonaws.com/?list-type=2"
                   f"&prefix={date}/{run}z/{path}/{stream}/&max-keys=1000")
            for attempt in range(6):
                try:
                    xml = urllib.request.urlopen(url, timeout=120).read().decode()
                    break
                except Exception:
                    if attempt == 5:
                        raise
                    time.sleep(2 * (attempt + 1))
            total += sum(int(x) for x in re.findall(r"<Size>(\d+)</Size>", xml))
    return total

probe = "20260823"
per_day = grib_bytes_for_day(probe)
print(f"{probe}: {per_day/1e12:.2f} TB published across 00z/06z/12z/18z")

n_days = sum(root[e][r]["time"].shape[0] for e in root.group_keys()
             for r in root[e].group_keys() if r == "00z")
print(f"\n{n_days} dates x ~1.77 TB/day (0p25) -> the ~1.9 PB in the table above,")
print("after allowing for the smaller 0.4-degree era (0.725 TB/day).")

20260823: 1.81 TB published across 00z/06z/12z/18z

1308 dates x ~1.77 TB/day (0p25) -> the ~1.9 PB in the table above,
after allowing for the smaller 0.4-degree era (0.725 TB/day).


### Date coverage — per cycle, across eras

A cycle's coverage is split across three era groups, because the model upgrades cut the
archive into `0p4` → `49r1` → `50r1`. To ask "how far back does 06z go?" you have to take
the union across eras, which is what the next cell does.

It also checks **contiguity**, and reports any missing day rather than only the endpoints —
a span of the right length says nothing about holes inside it.

> **Six or seven days really are missing, and they are not a pipeline failure.** ECMWF
> never published `20230427`–`20230502` at 0.4°, so no amount of re-running produces them.
> The window starts a day earlier (`20230426`) at 12z and 18z than at 00z and 06z, which
> is why the cycles differ by one date. Anything reporting these as gaps to be filled is
> reporting them wrong.

In [6]:
EPOCH = np.datetime64("1970-01-01")

print(f"{'cycle':<6} {'dates':>6}  {'span':<26} {'cal.days':>9} {'gaps':>5}")
for run in ("00z", "06z", "12z", "18z"):
    per_era = []
    for era in root.group_keys():
        if run in list(root[era].group_keys()):
            t = root[era][run]["time"][:]
            per_era.append(EPOCH + t.astype("timedelta64[h]"))
    d = np.unique(np.concatenate(per_era)).astype("datetime64[D]")

    span = int((d[-1] - d[0]).astype(int)) + 1
    every_day = np.arange(d[0], d[-1] + np.timedelta64(1, "D"))
    missing = sorted(set(every_day) - set(d))

    print(f"{run:<6} {len(d):>6}  {d[0]} .. {d[-1]} {span:>9} {len(missing):>5}")
    if missing:
        print(f"        never published upstream: {missing[0]} .. {missing[-1]}")

print("\nEras per cycle:")
for era in sorted(root.group_keys()):
    runs = sorted(root[era].group_keys())
    t = root[era][runs[0]]["time"][:]
    d = (EPOCH + t.astype("timedelta64[h]")).astype("datetime64[D]")
    print(f"  {era:<5} {d.min()} .. {d.max()}   cycles: {', '.join(runs)}")

cycle   dates  span                        cal.days  gaps
00z      1308  2023-01-18 .. 2026-08-23      1314     6
        never published upstream: 2023-04-27 .. 2023-05-02
06z      1308  2023-01-18 .. 2026-08-23      1314     6
        never published upstream: 2023-04-27 .. 2023-05-02
12z      1307  2023-01-18 .. 2026-08-23      1314     7
        never published upstream: 2023-04-26 .. 2023-05-02
18z      1307  2023-01-18 .. 2026-08-23      1314     7
        never published upstream: 2023-04-26 .. 2023-05-02

Eras per cycle:
  0p4   2023-01-18 .. 2024-02-28   cycles: 00z, 06z, 12z, 18z
  49r1  2024-02-29 .. 2026-05-12   cycles: 00z, 06z, 12z, 18z
  50r1  2026-05-13 .. 2026-08-23   cycles: 00z, 06z, 12z, 18z


## 2. One group as xarray

`chunks=None` is deliberate: the returned DataArrays are **not** dask arrays, so `.values`
on a single-message selection is a direct icechunk read with no dask graph in the way.

> **Longitude runs −180 → +179.75, ascending.** The ECMWF field starts at −180°E, not 0°.
> So `.sel(longitude=slice(15, 80))` for East Africa works directly, with no roll and no
> `sortby`. An earlier version of the builder assumed a 0° origin and every store built
> from it was labelled 180° out — subsets that looked plausible were reading the eastern
> Pacific. If you ever fork this code, assert the axis rather than trusting it.
>
> **Latitude DESCENDS** (+90 → −90), so a latitude slice goes `slice(north, south)`.

In [7]:
ERA, RUN = "50r1", "00z"   # stage 3 reads 00z; see section 5

ds = xr.open_zarr(session.store, group=f"{ERA}/{RUN}", consolidated=False,
                  zarr_format=3, decode_timedelta=True, chunks=None)

_h = (ds.step.values / np.timedelta64(1, "h")).astype(int)

print("dims      :", dict(ds.sizes))
print("latitude  :", float(ds.latitude[0]), "->", float(ds.latitude[-1]))
print("longitude :", float(ds.longitude[0]), "->", float(ds.longitude[-1]))
print("steps     :", f"{len(_h)}, +{_h[0]}h .. +{_h[-1]}h "
                     f"(3-hourly to +{_h[_h <= 144][-1]}h, then 6-hourly)")
print("\nsurface vars:", [v for v in ds.data_vars if "isobaricInhPa" not in ds[v].dims][:12])
print("level vars  :", [v for v in ds.data_vars if "isobaricInhPa" in ds[v].dims][:12])

assert float(ds.longitude[0]) == -180.0, "grid origin is not -180: do not trust subsets"
assert float(ds.latitude[0]) > float(ds.latitude[-1]), "latitude is not descending"

dims      : {'time': 103, 'number': 51, 'step': 85, 'latitude': 721, 'longitude': 1440, 'isobaricInhPa': 14}
latitude  : 90.0 -> -90.0
longitude : -180.0 -> 179.75
steps     : 85, +0h .. +360h (3-hourly to +144h, then 6-hourly)

surface vars: ['100u', 'ewss', '10fg', '2d', '100v', 'mucape', '10fg3', 'lsm', 'mn2t3', 'mn2t6', 'msl', 'asn']
level vars  : ['d', 'gh', 'r', 'q', 't', 'vo', 'u', 'v', 'z', 'w']


### Read one field — the unit of work

One `(variable, member, step)` selection is **one GRIB message**. That is the atom the
whole production DAG is built from: one task = one message.

Pick a date that exists in the group. The published store's tip moves as the archive is
extended, so take it from the data rather than hard-coding one.

In [ ]:
# The EA / SW-IO box -- the domain the realized stores use.
LAT_MAX, LAT_MIN = 40.0, -40.0     # latitude DESCENDS
LON_MIN, LON_MAX = 15.0, 80.0

DATE = ds.time.values[-1]          # newest date in this group
print("date:", str(DATE)[:10])

import time
t0 = time.time()
field = (ds["t2m"]
         .sel(time=DATE, number=0, step=np.timedelta64(0, "h"))
         .sel(latitude=slice(LAT_MAX, LAT_MIN), longitude=slice(LON_MIN, LON_MAX)))
arr = np.asarray(field.values, dtype=np.float32)
print(f"{arr.shape}  {time.time()-t0:.1f}s  "
      f"finite={np.isfinite(arr).mean():.3f}  mean={np.nanmean(arr):.2f} K")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(arr - 273.15, origin="upper", cmap="RdYlBu_r",
               extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX])
ax.set_title(f"2 m temperature (degC) - control member, +0h\n{str(DATE)[:10]} {ERA}/{RUN}")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

### The ensemble is real — check it rather than assume it

The store keeps **every raw member** (`number = 0..50`, control at 0); nothing is averaged
anywhere in the pipeline. The cheap proof that a member selection is not silently
broadcasting the same field 51 times is physics: **spread grows with lead time**. A
broadcast bug cannot fake that.

Use an upper-air field such as `gh500`. Do not use a 2 m temperature field for this — flat
tropical spread reads as a failure when it is nothing of the sort.

In [ ]:
MEMBERS = 5     # keep the demo small: 5 members x 2 steps = 10 messages

gh = ds["gh"].sel(isobaricInhPa=500)   # level FIRST, while still lazily indexed
for step_h in (0, 120):
    box = (gh.sel(time=DATE, step=np.timedelta64(step_h, "h"))
             .sel(number=slice(0, MEMBERS - 1),
                  latitude=slice(LAT_MAX, LAT_MIN),
                  longitude=slice(LON_MIN, LON_MAX)))
    v = np.asarray(box.values, dtype=np.float32)
    print(f"+{step_h:>3}h  members={v.shape[0]}  spread(sd over members)={v.std(axis=0).mean():.4f}")

print("\nspread should grow with lead time; flat spread means a broadcast bug")

## 3. Realizing by hand

**Realization = decoding the virtual references into actual numbers for a chosen
box/variable subset, and storing them.** The virtual store is for exploring the whole
archive cheaply; a realized store is what you want for training or repeated analysis,
because every read of the virtual store re-fetches from AWS.

Here is the whole idea at a scale that runs in a notebook — a few variables, a few
members, a few steps, written to a **local** Icechunk store so it needs no credentials.

In [ ]:
# (source_var, level|None, output_name) -- the same shape as the production table
DEMO_CHANNELS = [
    ("tp",   None, "tp"),
    ("t2m",  None, "t2m"),
    ("gh",   500,  "gh500"),
    ("q",    850,  "q850"),
]
DEMO_MEMBERS = 3
DEMO_STEPS   = [0, 6, 12]

n_msg = len(DEMO_CHANNELS) * DEMO_MEMBERS * len(DEMO_STEPS)
print(f"{n_msg} GRIB messages  (a full production day is 38 x 51 x 53 = 102,714)")

In [ ]:
def read_message(var, level, date, member, step_h):
    """ONE message -> the box as float32. The unit of work.

    Level is selected FIRST, while the array is still lazily indexed, so the
    composed index stays narrow.
    """
    da = ds[var]
    if level is not None:
        da = da.sel(isobaricInhPa=level)
    da = da.sel(time=date, number=member, step=np.timedelta64(step_h, "h"),
                latitude=slice(LAT_MAX, LAT_MIN), longitude=slice(LON_MIN, LON_MAX))
    return np.asarray(da.values, dtype=np.float32)


t0 = time.time()
realized = {}
for src, lvl, out in DEMO_CHANNELS:
    stack = np.stack([[read_message(src, lvl, DATE, m, s) for m in range(DEMO_MEMBERS)]
                      for s in DEMO_STEPS])                      # (step, number, y, x)
    realized[out] = stack
    print(f"  {out:<6} {stack.shape}  {stack.nbytes/1e6:.1f} MB  "
          f"finite={np.isfinite(stack).mean():.3f}")
print(f"\n{n_msg} messages in {time.time()-t0:.0f}s")

### Write it to a local Icechunk store

This is a realized store in miniature: real decoded numbers, no virtual references, opens
with no network at all.

In [ ]:
import tempfile, pathlib

out_dir = pathlib.Path(tempfile.mkdtemp(prefix="demo-realized-"))
sink = icechunk.Repository.create(icechunk.local_filesystem_storage(str(out_dir)))
s = sink.writable_session("main")

lat = ds.latitude.sel(latitude=slice(LAT_MAX, LAT_MIN)).values
lon = ds.longitude.sel(longitude=slice(LON_MIN, LON_MAX)).values

out = xr.Dataset(
    {name: (("step", "number", "latitude", "longitude"), a) for name, a in realized.items()},
    coords={"step": np.array(DEMO_STEPS, "timedelta64[h]"),
            "number": np.arange(DEMO_MEMBERS), "latitude": lat, "longitude": lon},
    attrs={"source": f"{SRC_PREFIX} {ERA}/{RUN}", "date": str(DATE)[:10]},
)
out.to_zarr(s.store, mode="w", consolidated=False, zarr_format=3)
print("committed:", s.commit(f"demo realization {str(DATE)[:10]}"))

back = xr.open_zarr(sink.readonly_session("main").store, consolidated=False, zarr_format=3)
print(back)

## 4. The production path — one day with Frisky

The loop above is the right shape and the wrong scale. A production day is
**38 channels × 51 members × 53 steps ≈ 102,714 messages**, ~26.6 GB of float32, and it
pulls ~77 GB from AWS because a GRIB message is atomic. That needs a cluster.

`frisky_daily_dag.py` builds it as a futures DAG across six worker VMs. The shape, per date:

```
    read(m=0,  step=s) ---.        m=0 is the CONTROL
    read(m=1,  step=s) ----+--> stack_members(step=s) --> write_block -> chunk
    ...                   /                                 (number, y, x)
    read(m=50, step=s) --'
    ... one such fan-in per (channel, step) ...
```

Every leaf is independent and the only fan-in is a **stack over `number`, not a
reduction** — the store keeps every raw member, because what a downstream model trains on
is the spread itself. A `reduce_members` mean/sd fold exists but is probe-only, and the
writers cannot reach it.

### Running it

```bash
P=.venv/bin/python

# no network at all -- just exercise the DAG shape
$P frisky_daily_dag.py --synthetic --channels 4 --members 6 --steps 3

# smallest real thing
$P frisky_daily_dag.py --date 2026-08-23 --channels 1 --members 4 --steps 2

# one full day, written to a store
$P parallel_write.py --sink ea-swio/one-date --date 2026-08-23 \
     --channels 38 --members 51 --steps 53 --fork-once --env .env
```

Writing needs the EWC Ceph keys (`AK`/`SK` in `.env`); **reading the source never does**.
Workers are never given the keys — the client opens the sink, calls `session.fork()`, and
the resulting `ForkSession` carries its own storage config to each worker as a task
argument.

## 5. Choosing the variable set

Both `frisky_daily_dag.py` and `parallel_write.py` take **`--vars`**, which picks channels
by output name instead of taking the first `--channels` of the table:

```bash
$P parallel_write.py --sink ea-swio/rain-only --date 2026-08-23 \
     --vars tp tcwv q850 cape w700 --members 51 --steps 53 --env .env
```

An unknown name is refused up front, listing what is available — rather than failing
inside a worker forty minutes in.

### Adding a channel that is not in the table

Append a `(source_var, level|None, output_name)` triple to `CHANNELS` in
`frisky_daily_dag.py`, then **redeploy the workers**:

```bash
$D deploy_workers.py --install     # workers hold their own copy of the module
```

Skip that and they keep cutting the old box, and `write_block` fails on a shape mismatch.

Three constraints on what you can add:

In [ ]:
# 1. The variable must exist in the era you are building.
#    `w` and CAPE exist in 49r1/50r1 but NOT in 0p4 -- which is why a 0p4 season
#    is a 36-channel store and 49r1/50r1 are 38.
#
#    NOTE `cape` printing ABSENT below is CORRECT, not a fault: from 2024-11-13
#    ECMWF publishes the field as `mucape`. That is the rename described under
#    "Two traps" -- and it is why the channel table stores a sentinel and
#    resolves the real source name per date.
print("vars in this group:", len(list(ds.data_vars)))
for probe in ("w", "mucape", "cape", "tp", "vo"):
    print(f"  {probe:<8} {'present' if probe in ds.data_vars else 'ABSENT'}")

# 2. A pressure level must be one the era actually carries.
print("\nlevels:", ds.isobaricInhPa.values)

# 3. Steps: a short cycle has 49, not 85. `--steps 53` over-runs 06z/18z.
print(f"\n{ERA}/{RUN} has {ds.sizes['step']} steps "
      f"(00z/12z: 85 to +360h, 06z/18z: 49 to +144h)")

### Two traps worth stating plainly

**ECMWF renamed the CAPE field.** Before `2024-11-13` it is `cape`; from that date it is
`mucape` (most-unstable parcel). They are one channel to the model, so the pipeline stores
both under the single name `cape` and resolves the source name *per date* at read time. If
you add a date-dependent variable, it needs the same treatment — a channel table has no
date in it. That constant was wrong by seven weeks until 2026-08-27, and the failure mode
is silent: an unwritten chunk reads back as NaN rather than raising.

**Stage ③ reads `00z` only.** `_open_era()` opens `group=f"{era}/00z"`, hard-coded — there
is no `--run` flag anywhere in the realization stage. The virtual store carries all four
cycles; the realized stores are built from `00z`. Pointing it at `06z` or `18z` is not just
a matter of parameterising that string: those groups have a 49-step axis, so the default
`--steps 53` would over-run it.

### Verifying a realized store

Never trust "0 failed blocks":

```bash
$P check_chunks.py  --prefix ea-cgan/<sink> --env .env              # reads the manifest
$P check_members.py --prefix ea-cgan/<sink> --env .env --var gh500  # spread grows?
```

A hole is loud: unwritten chunks read back as NaN (`finite 0.000`), because the schema is
written metadata-only with a NaN fill value. There is no zero-fill to hide behind.

**Always pass `--var gh500` to `check_members.py`.** With no `--var` it takes the first
data variable, and if that lands on a 2 m temperature field the flat tropical spread reads
as a failure.

---

## Where the pieces live

| | |
|---|---|
| `../RUNBOOK.md` | the whole three-stage pipeline |
| `OPENING_PUBLISHED_ECMWF_ICECHUNK.md` | reading the published store |
| `frisky_daily_dag.py` | the DAG, the domain, the channel table |
| `parallel_write.py` / `build_corpus.py` | one date / a season |
| `SCALING.md` | why ~96 concurrent readers, and not more |
| `../2-icechunk-virtual/` | how the virtual store is built |